In [5]:
# import libraries
import pandas as pd
from datetime import datetime
from time_diff_function import timediff

In [6]:
start_time  = datetime.now()
print(f'Getting \JSE\MTM\.. databefore StatPro run ...')

Getting \JSE\MTM\.. databefore StatPro run ...


In [7]:
# get instruments.csv and Holdings.csv data in eagle folder
jse_path = r'P:\Investments\Bonds\Fixed Interest Model Portfolios\Bond funds month end credit info\Template'
jse_data = pd.read_excel(jse_path + r'\All Models Live.xlsm', sheet_name = 'JSE MTM')

In [4]:
# get lookups for missing Instruments.csv and Holdings.csv data
py_reports   = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
currency     = pd.read_excel(py_reports, sheet_name = 'statpro', usecols = "A:B")
# for "NA" passing as NaN - https://stackoverflow.com/questions/41417214/prevent-pandas-from-reading-na-as-nan

# get issuer names and shorter issuer name replacements
issuer_name  = pd.read_excel(py_reports, sheet_name = 'statpro', usecols = "E:F").dropna(axis = 0, how = 'all')

In [5]:
# function to get country bigramme given currency trigramme
def cntry(curr):
    if currency['CURRENCY_CODE'].isin([curr]).any():
        return currency[currency['CURRENCY_CODE'] == curr].iat[0, 1]
    else:
        return 'US'
    
#cntry('NAD') # test the function

In [6]:
# function to shorten issuer name to 50 characters
def issuer(txt):
    if issuer_name['ISSUER_LONG_NAME'].isin([txt]).any():
        return issuer_name[issuer_name['ISSUER_LONG_NAME'] == txt].iat[0,1]
    else:
        return txt[:50]

# test the function
text1 = 'TOM BURKE COMMUNITY TRUST INVESTMENT SPV (PTY) LTD RF'
text2 = '123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890'
#print(issuer(text1), len(text1), len(issuer(text1)))
#print(issuer(text2), len(text2), len(issuer(text2)))

In [7]:
# identify the empty entries in the "COUNTRY_CODE" column ("NA" for Namibia returns a NaN)
k = instruments.loc[(instruments['COUNTRY_CODE'].isnull())  &
                    (instruments['CURRENCY_CODE'] != 'NAD') &
                    (instruments['ISSUE_NAME']    != 'NAMIBIA')]

# identify the > 50 ISSUE_NAME securities in Instruments.csv
p = instruments[instruments['ISSUE_NAME'].str.len() > 50]

# identify the > 50 ISSUERCODE securities in Holdings.csv
q = holdings[holdings['ISSUERCODE'].str.len() > 50]

# summarise missing data
print('(1) ' + str(len(k)) + ' instruments with empty COUNTRY_CODE')
print('(2) ' + str(len(p)) + ' instruments with ISSUE_NAME over 50')
print('(3) ' + str(len(q)) + ' holding with ISSUERCODE over 50')

(1) 0 instruments with empty COUNTRY_CODE
(2) 0 instruments with ISSUE_NAME over 50
(3) 0 holding with ISSUERCODE over 50


In [8]:
# (1) update missing instruments COUNTRY_CODE
if len(k) > 0:
    for row_number in range(len(k)):
        instruments.at[k.index[row_number], 'COUNTRY_CODE'] = cntry(instruments.at[k.index[row_number], 'CURRENCY_CODE'])
    # check that COUNTRY_CODE was updated
    instruments[k.index[0]:k.index[len(k) - 1] + 1]

In [9]:
# (2) update long, i.e., > 50 characters, ISSUE_NAME in instruments dataframe
if len(p) > 0:
    for row_number in range(len(p)):
        instruments.at[p.index[row_number], 'ISSUE_NAME'] = issuer(instruments.at[p.index[row_number], 'ISSUE_NAME'])
    # check that ISSUE_NAME in instruments was updated
    instruments[p.index[0]:p.index[len(p) - 1] + 1]

In [10]:
# (3) update long, i.e., > 50 characters, ISSUERNAME in holdings dataframe
if len(q) > 0:
    for row_number in range(len(q)):
        holdings.at[q.index[row_number], 'ISSUERCODE'] = issuer(holdings.at[q.index[row_number], 'ISSUERCODE'])
    # check that ISSUERCODE in holdings was updated
    holdings[q.index[0]:q.index[len(q) - 1] + 1]

In [11]:
# save the datframes over the Instruments.csv and Holdings.csv files in the eagle folder
#https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html
instruments.to_csv(eagle_path + r'\Instruments.csv', index  = False)
holdings.to_csv(   eagle_path + r'\Holdings.csv'   , index  = False)

In [12]:
print(f'Amending \eagle\.. files before StatPro run completed: {timediff(start_time, datetime.now())}')

Amending \eagle\.. files before StatPro run completed: 0d 0hr 0min 5.0sec
